In [ ]:
from impact import Impact
from impact.model.model import LUMEImpactModel
from impact.model.config import VariableMappingConfig
from lume.variables import ScalarVariable

In [ ]:
# Load LCLS Impact-T lattice
imp = Impact("templates/lcls_injector/ImpactT.in")

# Automatically convert to lume-model
ele_name_mappings = {"QUAD:IN20:425": "QE01"}
model = LUMEImpactModel.from_impact(
    imp,
    variable_mapping=VariableMappingConfig(element_pattern="{name}:{attrib}"),
    ele_name_mappings=ele_name_mappings,
    ele_regex_override=r"^(?P<name>.+):(?P<attrib>[^:]+)$",  # Regex to capture item after last colon as attribute, everything else as name
    dummy_run=True,
)

# Show default loaded variables, can control names and inclusion of all attributes from `VariableMappingConfig` object
[var for var in model.vars if "IN20" in var.name]

In [ ]:
# Add custom variable for each quad (SLAC BCTRL)
inv_name_map = {v: k for k, v in ele_name_mappings.items()}
ctrl_vars = []
for ele in imp.lattice:
    name = ele["name"]
    if ele["type"] == "quadrupole":
        ctrl_vars.append(
            ScalarVariable(name=inv_name_map.get(name, name) + ":BCTRL", unit="kG")
        )
model.vars.extend(ctrl_vars)


# Add custom handlers using element matching
@model.transformer.ele_setter(attrib="BCTRL", ele_type="quadrupole")
def set_bctrl(imp, value, name, mapped_name, attrib):
    b1_grad = (
        value / imp.ele[mapped_name]["L"]
    )  # Example calc, use own model to convert
    print(f"Setting {name} ({mapped_name}) b1_grad = {b1_grad:.2f}")


@model.transformer.ele_getter(attrib="BCTRL", ele_type="quadrupole")
def get_bctrl(imp, name, mapped_name, attrib):
    b1_grad = (
        imp.ele[mapped_name]["b1_gradient"] * imp.ele[mapped_name]["L"]
    )  # Example calc, use own model to convert
    print(f"Read {name} ({mapped_name}) b1_grad = {b1_grad:.2f}")


# Try using the setter
model.set({"QUAD:IN20:425:BCTRL": 0.0})